# 周末定价策略对比: max(0,6) vs max(2,6)

对比两种参照物选择策略:
- **max(hour_0, hour_6)**: 0点和6点USDCNH取较大值
- **max(hour_2, hour_6)**: 2点和6点USDCNH取较大值

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import os
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = [14, 8]

OUTPUT_DIR = r"c:\Users\tencentren\CodeBuddy\FX_SYSTEM\bmad-quant-system\output"
JPY_WEEKLY_VOLUME = 4_000_000_000

print("库导入完成!")

In [ ]:
# 加载数据
all_files = os.listdir(OUTPUT_DIR)
jpycnh_files = [f for f in all_files if 'JPYCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]
usdcnh_files = [f for f in all_files if 'USDCNH' in f and '1year' in f and f.endswith('.xlsx') and not f.startswith('~$')]

print(f"加载 JPYCNH: {jpycnh_files[0]}")
df_jpycnh = pd.read_excel(os.path.join(OUTPUT_DIR, jpycnh_files[0]))
df_jpycnh['timestamp'] = pd.to_datetime(df_jpycnh['timestamp'])
df_jpycnh.set_index('timestamp', inplace=True)

print(f"加载 USDCNH: {usdcnh_files[0]}")
df_usdcnh = pd.read_excel(os.path.join(OUTPUT_DIR, usdcnh_files[0]))
df_usdcnh['timestamp'] = pd.to_datetime(df_usdcnh['timestamp'])
df_usdcnh.set_index('timestamp', inplace=True)

print(f"\nJPYCNH: {df_jpycnh.shape}, 日期: {df_jpycnh.index.min()} ~ {df_jpycnh.index.max()}")
print(f"USDCNH: {df_usdcnh.shape}, 日期: {df_usdcnh.index.min()} ~ {df_usdcnh.index.max()}")

In [ ]:
# 准备周六数据
for df in [df_jpycnh, df_usdcnh]:
    df['weekday'] = df.index.dayofweek
    df['date'] = df.index.date
    df['hour'] = df.index.hour
    if 'mid' not in df.columns:
        df['mid'] = (df['bid'] + df['ask']) / 2

# 筛选周六 0-6点
jpycnh_sat = df_jpycnh[(df_jpycnh['weekday'] == 5) & (df_jpycnh['hour'] <= 6)].copy()
usdcnh_sat = df_usdcnh[(df_usdcnh['weekday'] == 5) & (df_usdcnh['hour'] <= 6)].copy()

common_dates = sorted(set(jpycnh_sat['date'].unique()) & set(usdcnh_sat['date'].unique()))
print(f"共同周六日期数: {len(common_dates)}")
print(f"日期范围: {common_dates[0]} ~ {common_dates[-1]}")

In [ ]:
def run_backtest(jpycnh_sat, usdcnh_sat, common_dates, strategy='max_0_6', spread_bps=5.0):
    """
    运行回测
    strategy: 'max_0_6' 或 'max_2_6'
    """
    spread_half = spread_bps / 2
    weekly_results = []
    
    for sat_date in common_dates:
        jpycnh_day = jpycnh_sat[jpycnh_sat['date'] == sat_date].copy()
        usdcnh_day = usdcnh_sat[usdcnh_sat['date'] == sat_date].copy()
        
        if jpycnh_day.empty or usdcnh_day.empty:
            continue
        
        # 获取各小时USDCNH
        usdcnh_h0 = usdcnh_day[usdcnh_day['hour'] == 0]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 0]) > 0 else np.nan
        usdcnh_h2 = usdcnh_day[usdcnh_day['hour'] == 2]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 2]) > 0 else np.nan
        usdcnh_h6 = usdcnh_day[usdcnh_day['hour'] == 6]['mid'].mean() if len(usdcnh_day[usdcnh_day['hour'] == 6]) > 0 else np.nan
        
        if pd.isna(usdcnh_h6):
            continue
        
        # 根据策略选择参照物
        if strategy == 'max_0_6':
            if pd.isna(usdcnh_h0):
                continue
            ref_usdcnh = max(usdcnh_h0, usdcnh_h6)
            ref_used = 'h0' if ref_usdcnh == usdcnh_h0 else 'h6'
        else:  # max_2_6
            if pd.isna(usdcnh_h2):
                continue
            ref_usdcnh = max(usdcnh_h2, usdcnh_h6)
            ref_used = 'h2' if ref_usdcnh == usdcnh_h2 else 'h6'
        
        # 对齐时间戳计算
        common_idx = jpycnh_day.index.intersection(usdcnh_day.index)
        if len(common_idx) < 10:
            continue
        
        jpycnh_aligned = jpycnh_day.loc[common_idx].copy()
        jpycnh_aligned['jpycnh_ret'] = jpycnh_aligned['mid'].pct_change() * 10000
        
        week_pnl_bps = 0
        week_trades = 0
        week_wins = 0
        
        for i in range(1, len(common_idx)):
            ts = common_idx[i]
            jpycnh_actual = jpycnh_aligned.loc[ts, 'jpycnh_ret']
            
            if pd.isna(jpycnh_actual):
                continue
            
            # 简化: 预测为0 (factor=0)
            quote_error = abs(jpycnh_actual)
            
            if quote_error < spread_half:
                pnl = spread_half
                week_wins += 1
            else:
                pnl = spread_half - quote_error
            
            week_pnl_bps += pnl
            week_trades += 1
        
        # 计算CNH损益
        jpycnh_rate = jpycnh_day['mid'].mean() / 100  # 转换为每1 JPY
        avg_pnl_bps = week_pnl_bps / week_trades if week_trades > 0 else 0
        week_pnl_cnh = avg_pnl_bps * 0.0001 * JPY_WEEKLY_VOLUME * jpycnh_rate
        
        weekly_results.append({
            'date': sat_date,
            'strategy': strategy,
            'ref_used': ref_used,
            'usdcnh_h0': usdcnh_h0,
            'usdcnh_h2': usdcnh_h2 if strategy == 'max_2_6' else np.nan,
            'usdcnh_h6': usdcnh_h6,
            'ref_usdcnh': ref_usdcnh,
            'trades': week_trades,
            'wins': week_wins,
            'win_rate': week_wins / week_trades if week_trades > 0 else 0,
            'avg_pnl_bps': avg_pnl_bps,
            'week_pnl_cnh': week_pnl_cnh,
        })
    
    return pd.DataFrame(weekly_results)

In [ ]:
# 运行两种策略回测 (使用5bps点差)
SPREAD_BPS = 5.0

print(f"运行回测 (点差={SPREAD_BPS}bps)...")
print("="*60)

# max(0,6) 策略
df_max_0_6 = run_backtest(jpycnh_sat, usdcnh_sat, common_dates, strategy='max_0_6', spread_bps=SPREAD_BPS)
print(f"\nmax(0,6) 策略:")
print(f"  周数: {len(df_max_0_6)}")
print(f"  总PnL: {df_max_0_6['week_pnl_cnh'].sum():,.0f} CNH")
print(f"  平均每周: {df_max_0_6['week_pnl_cnh'].mean():,.0f} CNH")
print(f"  平均胜率: {df_max_0_6['win_rate'].mean()*100:.1f}%")

# max(2,6) 策略
df_max_2_6 = run_backtest(jpycnh_sat, usdcnh_sat, common_dates, strategy='max_2_6', spread_bps=SPREAD_BPS)
print(f"\nmax(2,6) 策略:")
print(f"  周数: {len(df_max_2_6)}")
print(f"  总PnL: {df_max_2_6['week_pnl_cnh'].sum():,.0f} CNH")
print(f"  平均每周: {df_max_2_6['week_pnl_cnh'].mean():,.0f} CNH")
print(f"  平均胜率: {df_max_2_6['win_rate'].mean()*100:.1f}%")

In [ ]:
# 合并数据用于对比
df_max_0_6['cumsum_pnl'] = df_max_0_6['week_pnl_cnh'].cumsum()
df_max_2_6['cumsum_pnl'] = df_max_2_6['week_pnl_cnh'].cumsum()

# 对齐日期
common_weeks = set(df_max_0_6['date']) & set(df_max_2_6['date'])
df_0_6 = df_max_0_6[df_max_0_6['date'].isin(common_weeks)].copy().sort_values('date').reset_index(drop=True)
df_2_6 = df_max_2_6[df_max_2_6['date'].isin(common_weeks)].copy().sort_values('date').reset_index(drop=True)

# 重新计算累计
df_0_6['cumsum_pnl'] = df_0_6['week_pnl_cnh'].cumsum()
df_2_6['cumsum_pnl'] = df_2_6['week_pnl_cnh'].cumsum()

print(f"共同周数: {len(common_weeks)}")

## 回测对比图

In [ ]:
# 绘制对比图
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

dates = pd.to_datetime(df_0_6['date'])

# ========== 图1: 累计PnL对比 ==========
ax1 = axes[0, 0]
ax1.plot(dates, df_0_6['cumsum_pnl']/1000, 'b-', linewidth=2.5, marker='o', markersize=4, label='max(0,6)')
ax1.plot(dates, df_2_6['cumsum_pnl']/1000, 'r-', linewidth=2.5, marker='s', markersize=4, label='max(2,6)')
ax1.fill_between(dates, df_0_6['cumsum_pnl']/1000, df_2_6['cumsum_pnl']/1000, 
                 where=(df_0_6['cumsum_pnl'] > df_2_6['cumsum_pnl']),
                 color='blue', alpha=0.2, label='max(0,6)优势')
ax1.fill_between(dates, df_0_6['cumsum_pnl']/1000, df_2_6['cumsum_pnl']/1000, 
                 where=(df_0_6['cumsum_pnl'] < df_2_6['cumsum_pnl']),
                 color='red', alpha=0.2, label='max(2,6)优势')
ax1.set_xlabel('日期', fontsize=12)
ax1.set_ylabel('累计PnL (千CNH)', fontsize=12)
ax1.set_title(f'累计PnL对比 (点差={SPREAD_BPS}bps)', fontsize=14, fontweight='bold')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax1.xaxis.set_major_locator(mdates.WeekdayLocator(interval=4))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# ========== 图2: 每周PnL差异 ==========
ax2 = axes[0, 1]
diff_pnl = (df_0_6['week_pnl_cnh'] - df_2_6['week_pnl_cnh']) / 1000
colors = ['green' if x >= 0 else 'red' for x in diff_pnl]
ax2.bar(range(len(diff_pnl)), diff_pnl, color=colors, alpha=0.7, edgecolor='black', linewidth=0.5)
ax2.axhline(y=0, color='black', linestyle='-', linewidth=1)
ax2.axhline(y=diff_pnl.mean(), color='blue', linestyle='--', linewidth=2, label=f'平均差异: {diff_pnl.mean():.2f}K')
ax2.set_xlabel('周数', fontsize=12)
ax2.set_ylabel('PnL差异 (千CNH)', fontsize=12)
ax2.set_title('每周PnL差异 [max(0,6) - max(2,6)]', fontsize=14, fontweight='bold')
ax2.legend(loc='best', fontsize=10)
ax2.grid(True, alpha=0.3, axis='y')

# 添加统计
pos_count = (diff_pnl >= 0).sum()
neg_count = (diff_pnl < 0).sum()
ax2.text(0.98, 0.95, f'max(0,6)更优: {pos_count}周\nmax(2,6)更优: {neg_count}周', 
         transform=ax2.transAxes, fontsize=11, verticalalignment='top', horizontalalignment='right',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# ========== 图3: 每周PnL并排对比 ==========
ax3 = axes[1, 0]
x = np.arange(len(df_0_6))
width = 0.35
ax3.bar(x - width/2, df_0_6['week_pnl_cnh']/1000, width, label='max(0,6)', color='steelblue', alpha=0.8)
ax3.bar(x + width/2, df_2_6['week_pnl_cnh']/1000, width, label='max(2,6)', color='coral', alpha=0.8)
ax3.set_xlabel('周数', fontsize=12)
ax3.set_ylabel('每周PnL (千CNH)', fontsize=12)
ax3.set_title('每周PnL并排对比', fontsize=14, fontweight='bold')
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')
ax3.set_xticks(x[::4])
ax3.set_xticklabels([str(i+1) for i in x[::4]])

# ========== 图4: 汇总统计 ==========
ax4 = axes[1, 1]
ax4.axis('off')

# 计算统计数据
total_0_6 = df_0_6['week_pnl_cnh'].sum()
total_2_6 = df_2_6['week_pnl_cnh'].sum()
avg_0_6 = df_0_6['week_pnl_cnh'].mean()
avg_2_6 = df_2_6['week_pnl_cnh'].mean()
winrate_0_6 = df_0_6['win_rate'].mean() * 100
winrate_2_6 = df_2_6['win_rate'].mean() * 100
diff_total = total_0_6 - total_2_6
diff_pct = (total_0_6 / total_2_6 - 1) * 100 if total_2_6 != 0 else 0

summary_text = f"""
{'='*50}
回测对比汇总 (点差={SPREAD_BPS}bps, 周数={len(df_0_6)})
{'='*50}

                    max(0,6)        max(2,6)        差异
{'-'*50}
总PnL (CNH)      {total_0_6:>12,.0f}  {total_2_6:>12,.0f}  {diff_total:>+12,.0f}
平均每周 (CNH)    {avg_0_6:>12,.0f}  {avg_2_6:>12,.0f}  {avg_0_6-avg_2_6:>+12,.0f}
平均胜率 (%)      {winrate_0_6:>12.1f}  {winrate_2_6:>12.1f}  {winrate_0_6-winrate_2_6:>+12.1f}
{'-'*50}

max(0,6) 相对优势: {diff_pct:+.2f}%
年化差异 (52周): {(avg_0_6-avg_2_6)*52:+,.0f} CNH

结论: {'max(0,6) 更优' if diff_total > 0 else 'max(2,6) 更优'}
"""

ax4.text(0.05, 0.95, summary_text, transform=ax4.transAxes,
        fontsize=11, verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='lightgray', alpha=0.3))

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, f'strategy_comparison_max06_vs_max26_{SPREAD_BPS}bps.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f"\n图表已保存!")

## 详细数据对比

In [ ]:
# 创建详细对比表
comparison = pd.DataFrame({
    '日期': df_0_6['date'],
    'max(0,6)_PnL': df_0_6['week_pnl_cnh'],
    'max(2,6)_PnL': df_2_6['week_pnl_cnh'],
    '差异': df_0_6['week_pnl_cnh'] - df_2_6['week_pnl_cnh'],
    'max(0,6)_胜率': df_0_6['win_rate'] * 100,
    'max(2,6)_胜率': df_2_6['win_rate'] * 100,
    'max(0,6)_ref': df_0_6['ref_used'],
    'max(2,6)_ref': df_2_6['ref_used'],
})

comparison['差异'] = comparison['差异'].round(2)
comparison['更优策略'] = comparison['差异'].apply(lambda x: 'max(0,6)' if x > 0 else ('max(2,6)' if x < 0 else '相同'))

print("每周详细对比:")
display(comparison)

In [ ]:
# 保存结果
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = os.path.join(OUTPUT_DIR, f'strategy_comparison_max06_vs_max26_{timestamp}.xlsx')

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    comparison.to_excel(writer, sheet_name='Weekly_Comparison', index=False)
    df_0_6.to_excel(writer, sheet_name='max_0_6_detail', index=False)
    df_2_6.to_excel(writer, sheet_name='max_2_6_detail', index=False)

print(f"结果已保存到: {output_file}")